In [5]:
from astroquery.jplhorizons import Horizons
from IPython.display import IFrame
import pandas as pd, base64, json

# get ephemeris with 3σ columns
h = Horizons(
    id="3I",   # technically 3I/ATLAS
    location="705",   # obs code
    epochs={"start":"2025-11-13 06:20", #ITS IN UTC
            "stop":"2025-11-13 06:30",
            "step":"1m"}  
    # hourly time + stepsize
)
df = (
    h.ephemerides().to_pandas().rename(columns={
        "RA":"RA_deg","DEC":"DEC_deg",
        "RA_3sigma":"RA3_asec","DEC_3sigma":"DEC3_asec"
    })
)

# center so aladin knows where to point and data array
ra0, dec0 = df.RA_deg.mean(), df.DEC_deg.mean()
records = df[["RA_deg","DEC_deg","datetime_str","RA3_asec","DEC3_asec"]].to_dict("records")
js_array = json.dumps(records)

# makes HTML page (why are markers so tough to display!!!)
html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Aladin Lite Unc</title>
  <script src="https://aladin.cds.unistra.fr/AladinLite/api/v3/latest/aladin.js"></script>
  <style>body,html {{margin:0;padding:0;height:100%;}}</style>
</head><body>
  <div id="aladin-lite-div" style="width:100%; height:100vh;"></div>
  <script>
  A.init.then(() => {{
    const aladin = A.aladin('#aladin-lite-div', {{
      survey:   'P/DSS2/color',
      target:   '{ra0:.6f} {dec0:.6f}',
      fov:      0.5,
      cooFrame: 'ICRS'
    }});

    // cross markers
    const trackLayer = A.catalog({{ name:'Track', shape:'cross', color:'magenta', sourceSize:10 }});
    aladin.addCatalog(trackLayer);

    // overlay for red ellipses
    const overlay = A.graphicOverlay({{ color:'red', lineWidth:2 }});
    aladin.addOverlay(overlay);

    const data = {js_array};

    data.forEach(d => {{
      // magenta cross
      trackLayer.addSources([
        A.marker(d.RA_deg, d.DEC_deg, {{
          popupTitle: d.datetime_str,
          popupDesc:  'σ_RA='+(d.RA3_asec/3600).toFixed(4)+'°, σ_Dec='+(d.DEC3_asec/3600).toFixed(4)+'°'
        }})
      ]); 

      // red uncertainty ellipse
      const raSig  = d.RA3_asec/3600;
      const decSig = d.DEC3_asec/3600;
      const ell = A.ellipse(d.RA_deg, d.DEC_deg, raSig, decSig, 0, {{}});
      overlay.add(ell);
    }});
   // rotate and flip
    const controls = document.createElement('div');
    controls.style.position = 'absolute';
    controls.style.top = '10px';
    controls.style.right = '10px';
    controls.style.background = 'rgba(255,255,255,0.8)';
    controls.style.padding = '8px';
    controls.innerHTML = `
      <label>Rotate: <input type="range" id="rot-slider" min="0" max="360" value="0"></label><br>
      <label><input type="checkbox" id="flip-toggle"> Flip Horizontally</label>
    `;
    document.body.appendChild(controls);

    const rotSlider = document.getElementById('rot-slider');
    const flipToggle = document.getElementById('flip-toggle');

    rotSlider.addEventListener('input', () => {{
      const angle = parseInt(rotSlider.value);
      aladin.setRotation(angle);
    }});
    flipToggle.addEventListener('change', () => {{
      const flipped = flipToggle.checked;
      document.getElementById('aladin-lite-div').style.transform = flipped ? 'scaleX(-1)' : '';
    }});
    // 

  }});
  </script>
</body></html>
"""

# embedding via base 64 and Iframe cause it helps (somehow)
b64 = base64.b64encode(html.encode('utf-8')).decode('ascii')
uri = 'data:text/html;base64,' + b64
IFrame(src=uri, width=800, height=600)


##### 3I, asteroid, fragment 
##### write up how this works, what stuff you need to provide, providing 
##### check if uncertainty ellipse works 